In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import pandas as pd

In [ ]:
def analyze_csv_files(root_dir):
    min_rows = float('inf')
    max_rows = float('-inf')

    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith(".csv"):
                filepath = os.path.join(dirpath, filename)
                try:
                    with open(filepath, 'r') as f:
                        rows = sum(1 for _ in f)
                        min_rows = min(min_rows, rows)
                        max_rows = max(max_rows, rows)
                except Exception as e:
                    print(f"Error processing file {filepath}: {e}")

    if min_rows == float('inf'):
        print("No .csv files found.")
    else:
        print(f"Minimum number of rows in .csv files: {min_rows}")
        print(f"Maximum number of rows in .csv files: {max_rows}")

# Example usage (replace with your actual path)
root_directory = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize"
analyze_csv_files(root_directory)

Minimum number of rows in .csv files: 31
Maximum number of rows in .csv files: 14348


In [ ]:
def list_files_with_few_rows(root_dir, max_rows=50):
    for dirpath, dirnames, filenames in os.walk(root_dir):
        for filename in filenames:
            if filename.endswith(".csv"):
                filepath = os.path.join(dirpath, filename)
                try:
                    with open(filepath, 'r') as f:
                        rows = sum(1 for _ in f)
                        if rows < max_rows:
                            print(f"File: {filepath}, Rows: {rows}")
                except Exception as e:
                    print(f"Error processing file {filepath}: {e}")

list_files_with_few_rows(root_directory)

In [ ]:
import os
import pandas as pd
import concurrent.futures

# Danh sách tên cột (mỗi keypoint có 4 cột: x, y, z, visibility)
columns_33_keypoints = [
    'NOSE_x', 'NOSE_y', 'NOSE_z', 'NOSE_visibility',
    'LEFT_EYE_INNER_x', 'LEFT_EYE_INNER_y', 'LEFT_EYE_INNER_z', 'LEFT_EYE_INNER_visibility',
    'LEFT_EYE_x', 'LEFT_EYE_y', 'LEFT_EYE_z', 'LEFT_EYE_visibility',
    'LEFT_EYE_OUTER_x', 'LEFT_EYE_OUTER_y', 'LEFT_EYE_OUTER_z', 'LEFT_EYE_OUTER_visibility',
    'RIGHT_EYE_INNER_x', 'RIGHT_EYE_INNER_y', 'RIGHT_EYE_INNER_z', 'RIGHT_EYE_INNER_visibility',
    'RIGHT_EYE_x', 'RIGHT_EYE_y', 'RIGHT_EYE_z', 'RIGHT_EYE_visibility',
    'RIGHT_EYE_OUTER_x', 'RIGHT_EYE_OUTER_y', 'RIGHT_EYE_OUTER_z', 'RIGHT_EYE_OUTER_visibility',
    'LEFT_EAR_x', 'LEFT_EAR_y', 'LEFT_EAR_z', 'LEFT_EAR_visibility',
    'RIGHT_EAR_x', 'RIGHT_EAR_y', 'RIGHT_EAR_z', 'RIGHT_EAR_visibility',
    'MOUTH_LEFT_x', 'MOUTH_LEFT_y', 'MOUTH_LEFT_z', 'MOUTH_LEFT_visibility',
    'MOUTH_RIGHT_x', 'MOUTH_RIGHT_y', 'MOUTH_RIGHT_z', 'MOUTH_RIGHT_visibility',
    'LEFT_SHOULDER_x', 'LEFT_SHOULDER_y', 'LEFT_SHOULDER_z', 'LEFT_SHOULDER_visibility',
    'RIGHT_SHOULDER_x', 'RIGHT_SHOULDER_y', 'RIGHT_SHOULDER_z', 'RIGHT_SHOULDER_visibility',
    'LEFT_ELBOW_x', 'LEFT_ELBOW_y', 'LEFT_ELBOW_z', 'LEFT_ELBOW_visibility',
    'RIGHT_ELBOW_x', 'RIGHT_ELBOW_y', 'RIGHT_ELBOW_z', 'RIGHT_ELBOW_visibility',
    'LEFT_WRIST_x', 'LEFT_WRIST_y', 'LEFT_WRIST_z', 'LEFT_WRIST_visibility',
    'RIGHT_WRIST_x', 'RIGHT_WRIST_y', 'RIGHT_WRIST_z', 'RIGHT_WRIST_visibility',
    'LEFT_PINKY_x', 'LEFT_PINKY_y', 'LEFT_PINKY_z', 'LEFT_PINKY_visibility',
    'RIGHT_PINKY_x', 'RIGHT_PINKY_y', 'RIGHT_PINKY_z', 'RIGHT_PINKY_visibility',
    'LEFT_INDEX_x', 'LEFT_INDEX_y', 'LEFT_INDEX_z', 'LEFT_INDEX_visibility',
    'RIGHT_INDEX_x', 'RIGHT_INDEX_y', 'RIGHT_INDEX_z', 'RIGHT_INDEX_visibility',
    'LEFT_THUMB_x', 'LEFT_THUMB_y', 'LEFT_THUMB_z', 'LEFT_THUMB_visibility',
    'RIGHT_THUMB_x', 'RIGHT_THUMB_y', 'RIGHT_THUMB_z', 'RIGHT_THUMB_visibility',
    'LEFT_HIP_x', 'LEFT_HIP_y', 'LEFT_HIP_z', 'LEFT_HIP_visibility',
    'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_HIP_z', 'RIGHT_HIP_visibility',
    'LEFT_KNEE_x', 'LEFT_KNEE_y', 'LEFT_KNEE_z', 'LEFT_KNEE_visibility',
    'RIGHT_KNEE_x', 'RIGHT_KNEE_y', 'RIGHT_KNEE_z', 'RIGHT_KNEE_visibility',
    'LEFT_ANKLE_x', 'LEFT_ANKLE_y', 'LEFT_ANKLE_z', 'LEFT_ANKLE_visibility',
    'RIGHT_ANKLE_x', 'RIGHT_ANKLE_y', 'RIGHT_ANKLE_z', 'RIGHT_ANKLE_visibility',
    'LEFT_HEEL_x', 'LEFT_HEEL_y', 'LEFT_HEEL_z', 'LEFT_HEEL_visibility',
    'RIGHT_HEEL_x', 'RIGHT_HEEL_y', 'RIGHT_HEEL_z', 'RIGHT_HEEL_visibility',
    'LEFT_FOOT_INDEX_x', 'LEFT_FOOT_INDEX_y', 'LEFT_FOOT_INDEX_z', 'LEFT_FOOT_INDEX_visibility',
    'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y', 'RIGHT_FOOT_INDEX_z', 'RIGHT_FOOT_INDEX_visibility'
]

# Các keypoint muốn loại bỏ hoàn toàn (theo chỉ số Mediapipe: 0 = NOSE, 1 = LEFT_EYE_INNER, ...)
unnecessary_points = {1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 17, 18, 19, 20, 21, 22, 29, 30, 31, 32}

def filter_columns_keep_xy(df, unnecessary_points, all_columns):
    """
    Trả về DataFrame chỉ giữ lại:
      - Với các keypoint không thuộc unnecessary_points: giữ lại cột x và y.
      - Với các keypoint thuộc unnecessary_points: loại bỏ toàn bộ (x, y, z, visibility).
    """
    columns_to_keep = []
    # Có 33 keypoint, mỗi keypoint có 4 cột: x, y, z, visibility
    for i in range(33):
        start_idx = i * 4    # cột x
        y_idx = i * 4 + 1    # cột y

        # Nếu keypoint i thuộc unnecessary_points => bỏ qua hoàn toàn
        if i in unnecessary_points:
            continue
        else:
            if start_idx < len(all_columns):
                columns_to_keep.append(all_columns[start_idx])
            if y_idx < len(all_columns):
                columns_to_keep.append(all_columns[y_idx])
    # Lọc lại DataFrame chỉ với các cột cần giữ (chỉ nếu cột tồn tại)
    existing_columns_to_keep = [c for c in columns_to_keep if c in df.columns]
    return df[existing_columns_to_keep]

def process_file(input_csv_path, output_csv_path):
    """
    Đọc file CSV từ input, lọc các cột theo yêu cầu rồi lưu vào output.
    """
    try:
        df = pd.read_csv(input_csv_path)
        df_filtered = filter_columns_keep_xy(df, unnecessary_points, columns_33_keypoints)
        df_filtered.to_csv(output_csv_path, index=False)
        print(f"Đã lưu file: {output_csv_path}")
    except pd.errors.EmptyDataError:
        print(f"File {input_csv_path} trống, bỏ qua.")
    except Exception as e:
        print(f"Lỗi xử lý file {input_csv_path}: {e}")

# Thư mục gốc chứa train/val/test (bên trong có các folder class) chứa file CSV
input_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split"
# Thư mục output
output_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]"
os.makedirs(output_folder, exist_ok=True)

# Sử dụng os.walk để duyệt mọi cấp thư mục và lưu đường dẫn cần xử lý
tasks = []
with concurrent.futures.ThreadPoolExecutor() as executor:
    for root, dirs, files in os.walk(input_folder):
        # Xây dựng đường dẫn tương đối cho thư mục hiện tại
        relative_path = os.path.relpath(root, input_folder)
        output_dir = os.path.join(output_folder, relative_path)
        os.makedirs(output_dir, exist_ok=True)

        for file_name in files:
            if file_name.endswith('.csv'):
                input_csv_path = os.path.join(root, file_name)
                output_csv_path = os.path.join(output_dir, file_name)
                # Gửi task xử lý file lên ThreadPoolExecutor
                tasks.append(executor.submit(process_file, input_csv_path, output_csv_path))

    # Đợi tất cả các task hoàn thành
    concurrent.futures.wait(tasks)


Đã lưu file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]/train/barbell biceps curl/barbell biceps curl_20.csv
Đã lưu file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]/train/barbell biceps curl/barbell biceps curl_27.csv
Đã lưu file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]/train/barbell biceps curl/barbell biceps curl_28.csv
Đã lưu file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]/train/barbell biceps curl/barbell biceps curl_39.csv
Đã lưu file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]/train/barbell biceps curl/barbell biceps curl_14.csv
Đã lưu file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]/train/barbell biceps curl/barbell biceps curl_31.csv
Đã lưu file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]/train/barbell biceps curl/barbell biceps curl_2.csv
Đã lưu file: /content/drive/MyDrive

In [ ]:
import os
import pandas as pd

def remove_z_and_confidence(df, x):
    columns_to_remove = []
    for col in x:
        if '_z' in col or '_visibility' in col:
            columns_to_remove.append(col)

    valid_columns_to_remove = [col for col in columns_to_remove if col in df.columns]
    df = df.drop(columns=valid_columns_to_remove)
    return df

# Đường dẫn thư mục
input_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_02"
output_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03"

# Danh sách các cột ban đầu
x = ['Frame', 'NOSE_x', 'NOSE_y', 'NOSE_z', 'NOSE_visibility', 'LEFT_EYE_INNER_x', 'LEFT_EYE_INNER_y', 'LEFT_EYE_INNER_z', 'LEFT_EYE_INNER_visibility', 'LEFT_EYE_x', 'LEFT_EYE_y', 'LEFT_EYE_z', 'LEFT_EYE_visibility', 'LEFT_EYE_OUTER_x', 'LEFT_EYE_OUTER_y', 'LEFT_EYE_OUTER_z', 'LEFT_EYE_OUTER_visibility', 'RIGHT_EYE_INNER_x', 'RIGHT_EYE_INNER_y', 'RIGHT_EYE_INNER_z', 'RIGHT_EYE_INNER_visibility', 'RIGHT_EYE_x', 'RIGHT_EYE_y', 'RIGHT_EYE_z', 'RIGHT_EYE_visibility', 'RIGHT_EYE_OUTER_x', 'RIGHT_EYE_OUTER_y', 'RIGHT_EYE_OUTER_z', 'RIGHT_EYE_OUTER_visibility', 'LEFT_EAR_x', 'LEFT_EAR_y', 'LEFT_EAR_z', 'LEFT_EAR_visibility', 'RIGHT_EAR_x', 'RIGHT_EAR_y', 'RIGHT_EAR_z', 'RIGHT_EAR_visibility', 'MOUTH_LEFT_x', 'MOUTH_LEFT_y', 'MOUTH_LEFT_z', 'MOUTH_LEFT_visibility', 'MOUTH_RIGHT_x', 'MOUTH_RIGHT_y', 'MOUTH_RIGHT_z', 'MOUTH_RIGHT_visibility', 'LEFT_SHOULDER_x', 'LEFT_SHOULDER_y', 'LEFT_SHOULDER_z', 'LEFT_SHOULDER_visibility', 'RIGHT_SHOULDER_x', 'RIGHT_SHOULDER_y', 'RIGHT_SHOULDER_z', 'RIGHT_SHOULDER_visibility', 'LEFT_ELBOW_x', 'LEFT_ELBOW_y', 'LEFT_ELBOW_z', 'LEFT_ELBOW_visibility', 'RIGHT_ELBOW_x', 'RIGHT_ELBOW_y', 'RIGHT_ELBOW_z', 'RIGHT_ELBOW_visibility', 'LEFT_WRIST_x', 'LEFT_WRIST_y', 'LEFT_WRIST_z', 'LEFT_WRIST_visibility', 'RIGHT_WRIST_x', 'RIGHT_WRIST_y', 'RIGHT_WRIST_z', 'RIGHT_WRIST_visibility', 'LEFT_PINKY_x', 'LEFT_PINKY_y', 'LEFT_PINKY_z', 'LEFT_PINKY_visibility', 'RIGHT_PINKY_x', 'RIGHT_PINKY_y', 'RIGHT_PINKY_z', 'RIGHT_PINKY_visibility', 'LEFT_INDEX_x', 'LEFT_INDEX_y', 'LEFT_INDEX_z', 'LEFT_INDEX_visibility', 'RIGHT_INDEX_x', 'RIGHT_INDEX_y', 'RIGHT_INDEX_z', 'RIGHT_INDEX_visibility', 'LEFT_THUMB_x', 'LEFT_THUMB_y', 'LEFT_THUMB_z', 'LEFT_THUMB_visibility', 'RIGHT_THUMB_x', 'RIGHT_THUMB_y', 'RIGHT_THUMB_z', 'RIGHT_THUMB_visibility', 'LEFT_HIP_x', 'LEFT_HIP_y', 'LEFT_HIP_z', 'LEFT_HIP_visibility', 'RIGHT_HIP_x', 'RIGHT_HIP_y', 'RIGHT_HIP_z', 'RIGHT_HIP_visibility', 'LEFT_KNEE_x', 'LEFT_KNEE_y', 'LEFT_KNEE_z', 'LEFT_KNEE_visibility', 'RIGHT_KNEE_x', 'RIGHT_KNEE_y', 'RIGHT_KNEE_z', 'RIGHT_KNEE_visibility', 'LEFT_ANKLE_x', 'LEFT_ANKLE_y', 'LEFT_ANKLE_z', 'LEFT_ANKLE_visibility', 'RIGHT_ANKLE_x', 'RIGHT_ANKLE_y', 'RIGHT_ANKLE_z', 'RIGHT_ANKLE_visibility', 'LEFT_HEEL_x', 'LEFT_HEEL_y', 'LEFT_HEEL_z', 'LEFT_HEEL_visibility', 'RIGHT_HEEL_x', 'RIGHT_HEEL_y', 'RIGHT_HEEL_z', 'RIGHT_HEEL_visibility', 'LEFT_FOOT_INDEX_x', 'LEFT_FOOT_INDEX_y', 'LEFT_FOOT_INDEX_z', 'LEFT_FOOT_INDEX_visibility', 'RIGHT_FOOT_INDEX_x', 'RIGHT_FOOT_INDEX_y', 'RIGHT_FOOT_INDEX_z', 'RIGHT_FOOT_INDEX_visibility']

# Duyệt qua tất cả các thư mục con (bao gồm cả 2 lớp hoặc nhiều hơn)
for root, dirs, files in os.walk(input_folder):
    for file_name in files:
        if file_name.endswith('.csv'):
            input_path = os.path.join(root, file_name)
            # Tính đường dẫn tương đối từ input_folder để tái tạo cấu trúc thư mục tương ứng trong output_folder
            relative_path = os.path.relpath(root, input_folder)
            output_dir = os.path.join(output_folder, relative_path)
            os.makedirs(output_dir, exist_ok=True)
            output_path = os.path.join(output_dir, file_name)
            try:
                df = pd.read_csv(input_path)
                df = remove_z_and_confidence(df, x)
                df.to_csv(output_path, index=False)
                print(f"Processed file: {output_path}")
            except pd.errors.EmptyDataError:
                print(f"File {file_name} is empty and will be skipped.")
            except Exception as e:
                print(f"Error processing file {file_name}: {e}")


Processed file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03/val/barbell biceps curl/barbell biceps curl_32.csv
Processed file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03/val/barbell biceps curl/barbell biceps curl_23.csv
Processed file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03/val/barbell biceps curl/barbell biceps curl_12.csv
Processed file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03/val/barbell biceps curl/barbell biceps curl_17.csv
Processed file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03/val/barbell biceps curl/barbell biceps curl_1.csv
Processed file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03/val/barbell biceps curl/barbell biceps curl_38.csv
Processed file: /content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_03/val/barbell biceps curl/barbell biceps curl_36.csv
Processed file: /content/drive/MyDr

In [ ]:
import os
import pandas as pd
import math
import numpy as np
import concurrent.futures

def calculate_angle(point1, mid_point, point3):
    x1, y1 = point1
    x2, y2 = mid_point
    x3, y3 = point3

    vec1 = (x2 - x1, y2 - y1)
    vec2 = (x2 - x3, y2 - y3)

    dot_product = vec1[0] * vec2[0] + vec1[1] * vec2[1]
    magnitude1 = math.sqrt(vec1[0]**2 + vec1[1]**2)
    magnitude2 = math.sqrt(vec2[0]**2 + vec2[1]**2)

    if magnitude1 == 0 or magnitude2 == 0:
        return np.nan

    cos_angle = dot_product / (magnitude1 * magnitude2)
    cos_angle = max(-1, min(1, cos_angle))
    angle_rad = math.acos(cos_angle)
    angle_deg = math.degrees(angle_rad)
    return angle_deg

def process_csv(filepath, output_filepath):
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return

    # File phải có 26 cột (13 điểm, mỗi điểm có 2 cột: x, y)
    if len(df.columns) != 26:
        print(f"File {filepath} không có đúng 26 cột.")
        return
    num_points = len(df.columns) // 2  # num_points = 13

    # Tính số góc kỳ vọng: với mỗi bộ ba điểm ta tính 3 góc
    expected_num_angles = 0
    for i in range(num_points - 2):
        for j in range(i + 1, num_points - 1):
            for k in range(j + 1, num_points):
                expected_num_angles += 3

    angles_data = []
    # Duyệt từng dòng (không tạo thêm cột frame)
    for idx, row in df.iterrows():
        row_angles = []
        for i in range(num_points - 2):
            for j in range(i + 1, num_points - 1):
                for k in range(j + 1, num_points):
                    def get_point(idx_point):
                        x_val = row.iloc[idx_point * 2]
                        y_val = row.iloc[idx_point * 2 + 1]
                        return (x_val, y_val)
                    p1 = get_point(i)
                    p2 = get_point(j)
                    p3 = get_point(k)
                    # Nếu một trong 3 điểm có giá trị trống, để cả 3 góc là NaN
                    if (pd.isna(p1[0]) or pd.isna(p1[1]) or
                        pd.isna(p2[0]) or pd.isna(p2[1]) or
                        pd.isna(p3[0]) or pd.isna(p3[1])):
                        row_angles.extend([np.nan, np.nan, np.nan])
                    else:
                        a1 = calculate_angle(p2, p1, p3)
                        a2 = calculate_angle(p1, p2, p3)
                        a3 = calculate_angle(p1, p3, p2)
                        row_angles.extend([a1, a2, a3])
        # Nếu vì lý do nào đó số góc tính được không đủ (thường đã đủ), bổ sung NaN
        if len(row_angles) < expected_num_angles:
            row_angles.extend([np.nan] * (expected_num_angles - len(row_angles)))
        angles_data.append(row_angles)

    columns = [f'Angle_{i+1}' for i in range(expected_num_angles)]
    angles_df = pd.DataFrame(angles_data, columns=columns)
    angles_df.to_csv(output_filepath, index=False)
    print(f"Angles saved to: {output_filepath}")

# Đường dẫn gốc input và output (với cấu trúc thư mục)
input_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]"
output_folder = "/content/drive/MyDrive/Gym/data/angles_output"
os.makedirs(output_folder, exist_ok=True)

def process_file(file_path):
    # Giữ nguyên cấu trúc thư mục ban đầu
    relative_path = os.path.relpath(os.path.dirname(file_path), input_folder)
    output_dir = os.path.join(output_folder, relative_path)
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, os.path.basename(file_path))
    process_csv(file_path, output_path)

# Thu thập tất cả các file CSV trong input_folder (bao gồm subfolder)
all_csv_files = []
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.csv'):
            all_csv_files.append(os.path.join(root, file))

num_workers = 32
with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
    executor.map(process_file, all_csv_files)


Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_18.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_27.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_30.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_22.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_39.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_4.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_61.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barbell biceps curl/barbell biceps curl_20.csv
Angles saved to: /content/drive/MyDrive/Gym/data/angles_output/train/barb

In [ ]:
import pandas as pd

# Danh sách tên của 13 điểm theo feature points của dự án
point_names = [
    "NOSE",
    "LEFT_SHOULDER",
    "RIGHT_SHOULDER",
    "LEFT_ELBOW",
    "RIGHT_ELBOW",
    "LEFT_WRIST",
    "RIGHT_WRIST",
    "LEFT_HIP",
    "RIGHT_HIP",
    "LEFT_KNEE",
    "RIGHT_KNEE",
    "LEFT_ANKLE",
    "RIGHT_ANKLE"
]

angle_mapping = []  # Danh sách chứa tuple: (Angle_Label, Description)
counter = 1

# Duyệt qua tất cả các bộ ba điểm (chỉ số tăng dần)
for i in range(len(point_names) - 2):
    for j in range(i + 1, len(point_names) - 1):
        for k in range(j + 1, len(point_names)):
            # Ba góc được tính cho bộ ba điểm (i, j, k)
            # Góc tại điểm i: tính từ hai điểm j và k
            label = f"Angle_{counter}"
            desc = f"Angle at {point_names[i]} (from {point_names[j]} and {point_names[k]})"
            angle_mapping.append((label, desc))
            counter += 1

            # Góc tại điểm j: tính từ hai điểm i và k
            label = f"Angle_{counter}"
            desc = f"Angle at {point_names[j]} (from {point_names[i]} and {point_names[k]})"
            angle_mapping.append((label, desc))
            counter += 1

            # Góc tại điểm k: tính từ hai điểm i và j
            label = f"Angle_{counter}"
            desc = f"Angle at {point_names[k]} (from {point_names[i]} and {point_names[j]})"
            angle_mapping.append((label, desc))
            counter += 1

# In ra 10 mapping đầu tiên để kiểm tra
for item in angle_mapping[:10]:
    print(item)

print(f"Tổng số góc tính được: {len(angle_mapping)}")  # Nhất thiết phải = 858

# Lưu mapping vào file CSV (nếu cần)
mapping_df = pd.DataFrame(angle_mapping, columns=["Angle_Label", "Description"])
mapping_df.to_csv("angle_mapping.csv", index=False)
print("Mapping saved to angle_mapping.csv")


('Angle_1', 'Angle at NOSE (from LEFT_SHOULDER and RIGHT_SHOULDER)')
('Angle_2', 'Angle at LEFT_SHOULDER (from NOSE and RIGHT_SHOULDER)')
('Angle_3', 'Angle at RIGHT_SHOULDER (from NOSE and LEFT_SHOULDER)')
('Angle_4', 'Angle at NOSE (from LEFT_SHOULDER and LEFT_ELBOW)')
('Angle_5', 'Angle at LEFT_SHOULDER (from NOSE and LEFT_ELBOW)')
('Angle_6', 'Angle at LEFT_ELBOW (from NOSE and LEFT_SHOULDER)')
('Angle_7', 'Angle at NOSE (from LEFT_SHOULDER and RIGHT_ELBOW)')
('Angle_8', 'Angle at LEFT_SHOULDER (from NOSE and RIGHT_ELBOW)')
('Angle_9', 'Angle at RIGHT_ELBOW (from NOSE and LEFT_SHOULDER)')
('Angle_10', 'Angle at NOSE (from LEFT_SHOULDER and LEFT_WRIST)')
Tổng số góc tính được: 858
Mapping saved to angle_mapping.csv


In [ ]:
import os
import pandas as pd
import math
import numpy as np
import concurrent.futures

def calculate_absolute_angle(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    if dx == 0:
        return 90.0
    angle_rad = math.atan(abs(dy/dx))
    return math.degrees(angle_rad)

def process_csv(filepath, output_filepath):
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return

    # Kiểm tra số cột: phải đúng 26 (13 điểm với 2 cột mỗi điểm)
    if len(df.columns) != 26:
        print(f"File {filepath} không có đúng 26 cột.")
        return

    num_points = 13

    # Số cặp điểm = C(13,2) = 78
    expected_num_angles = (num_points * (num_points - 1)) // 2

    angles_data = []
    # Duyệt từng dòng (mỗi dòng là 1 frame)
    for idx, row in df.iterrows():
        row_angles = []
        # Duyệt qua tất cả cặp điểm (i, j) với i < j
        for i in range(num_points - 1):
            for j in range(i + 1, num_points):
                # Lấy tọa độ điểm i và j
                p1 = (row.iloc[i * 2], row.iloc[i * 2 + 1])
                p2 = (row.iloc[j * 2], row.iloc[j * 2 + 1])
                # Nếu có giá trị bị thiếu thì góc = NaN
                if (pd.isna(p1[0]) or pd.isna(p1[1]) or
                    pd.isna(p2[0]) or pd.isna(p2[1])):
                    row_angles.append(np.nan)
                else:
                    angle = calculate_absolute_angle(p1, p2)
                    row_angles.append(angle)
        # Nếu số góc tính được chưa đủ (trường hợp hiếm gặp), bổ sung NaN
        if len(row_angles) < expected_num_angles:
            row_angles.extend([np.nan] * (expected_num_angles - len(row_angles)))
        angles_data.append(row_angles)

    # Tạo DataFrame kết quả với các cột Angle_1, Angle_2, ... Angle_78
    columns = [f"Angle_abs_{i+1}" for i in range(expected_num_angles)]
    angles_df = pd.DataFrame(angles_data, columns=columns)
    angles_df.to_csv(output_filepath, index=False)
    print(f"Angles saved to: {output_filepath}")

# Đường dẫn gốc input và output (giả sử cấu trúc thư mục có nhiều file CSV)
input_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]"
output_folder = "/content/drive/MyDrive/Gym/data/absolute_angles_output"
os.makedirs(output_folder, exist_ok=True)

def process_file(file_path):
    """
    Giữ nguyên cấu trúc thư mục ban đầu khi lưu file kết quả.
    """
    relative_path = os.path.relpath(os.path.dirname(file_path), input_folder)
    output_dir = os.path.join(output_folder, relative_path)
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, os.path.basename(file_path))
    process_csv(file_path, output_path)

# Thu thập tất cả các file CSV trong input_folder (bao gồm các subfolder)
all_csv_files = []
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.csv'):
            all_csv_files.append(os.path.join(root, file))

# Sử dụng multithreading để xử lý các file CSV song song
num_workers = 32
with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
    executor.map(process_file, all_csv_files)

#############################################
# Phần tạo mapping cho các góc tuyệt đối   #
#############################################

# Danh sách tên điểm theo thứ tự như trong file CSV
point_names = [
    "NOSE",
    "LEFT_SHOULDER",
    "RIGHT_SHOULDER",
    "LEFT_ELBOW",
    "RIGHT_ELBOW",
    "LEFT_WRIST",
    "RIGHT_WRIST",
    "LEFT_HIP",
    "RIGHT_HIP",
    "LEFT_KNEE",
    "RIGHT_KNEE",
    "LEFT_ANKLE",
    "RIGHT_ANKLE"
]

angle_mapping = []  # Danh sách chứa tuple: (Angle_Label, Description)
counter = 1

# Với 13 điểm, duyệt qua tất cả cặp điểm (không lặp lại)
for i in range(len(point_names) - 1):
    for j in range(i + 1, len(point_names)):
        label = f"Angle_abs_{counter}"
        desc = f"Absolute angle of the line between {point_names[i]} and {point_names[j]} relative to horizontal axis"
        angle_mapping.append((label, desc))
        counter += 1

# In ra 10 mapping đầu tiên để kiểm tra
for item in angle_mapping[:10]:
    print(item)

print(f"Tổng số góc tính được: {len(angle_mapping)}")  # Nhất thiết phải = 78

# Lưu mapping vào file CSV (nếu cần)
mapping_df = pd.DataFrame(angle_mapping, columns=["Angle_abs_Label", "Description"])
mapping_df.to_csv("/content/drive/MyDrive/Gym/data/Mapping csv/absolute_angle_mapping.csv", index=False)
print("Mapping saved to /content/drive/MyDrive/Gym/data/Mapping csv/absolute_angle_mapping.csv")


Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_39.csv
Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_22.csv
Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_27.csv
Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_35.csv
Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_37.csv
Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_34.csv
Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_25.csv
Angles saved to: /content/drive/MyDrive/Gym/data/absolute_angles_output/train/barbell biceps curl/barbell biceps curl_5.csv
A

In [ ]:
import os
import pandas as pd
import math
import numpy as np
import concurrent.futures

def calculate_absolute_angle(p1, p2):
    dx = p2[0] - p1[0]
    dy = p2[1] - p1[1]
    if dx == 0:
        return 90.0
    angle_rad = math.atan(abs(dy/dx))
    return math.degrees(angle_rad)

calculate_absolute_angle([885, 877],[899, 891])

45.0

In [5]:
import os
import pandas as pd
import concurrent.futures

def convert_to_relative(df):
    """
    Hàm chuyển đổi tọa độ tuyệt đối thành tọa độ tương đối dựa trên điểm NOSE.
    Giả sử df có 26 cột với thứ tự:
      NOSE_x, NOSE_y,
      LEFT_SHOULDER_x, LEFT_SHOULDER_y,
      RIGHT_SHOULDER_x, RIGHT_SHOULDER_y,
      LEFT_ELBOW_x, LEFT_ELBOW_y,
      RIGHT_ELBOW_x, RIGHT_ELBOW_y,
      LEFT_WRIST_x, LEFT_WRIST_y,
      RIGHT_WRIST_x, RIGHT_WRIST_y,
      LEFT_HIP_x, LEFT_HIP_y,
      RIGHT_HIP_x, RIGHT_HIP_y,
      LEFT_KNEE_x, LEFT_KNEE_y,
      RIGHT_KNEE_x, RIGHT_KNEE_y,
      LEFT_ANKLE_x, LEFT_ANKLE_y,
      RIGHT_ANKLE_x, RIGHT_ANKLE_y
    Trừ tọa độ NOSE (NOSE_x, NOSE_y) cho từng landmark còn lại.
    Kết quả là DataFrame mới chỉ chứa 24 cột (12 điểm x 2 tọa độ).
    """
    # Lấy tọa độ NOSE
    nose_x = df["NOSE_x"]
    nose_y = df["NOSE_y"]

    # Danh sách các landmark (ngoại trừ NOSE)
    landmarks = [
        "LEFT_SHOULDER", "RIGHT_SHOULDER",
        "LEFT_ELBOW", "RIGHT_ELBOW",
        "LEFT_WRIST", "RIGHT_WRIST",
        "LEFT_HIP", "RIGHT_HIP",
        "LEFT_KNEE", "RIGHT_KNEE",
        "LEFT_ANKLE", "RIGHT_ANKLE"
    ]

    # Tạo dictionary chứa tọa độ tương đối cho mỗi landmark
    relative_data = {}
    for lm in landmarks:
        # Tọa độ tương đối = tọa độ landmark - tọa độ NOSE
        relative_data[f"{lm}_x"] = df[f"{lm}_x"] - nose_x
        relative_data[f"{lm}_y"] = df[f"{lm}_y"] - nose_y

    new_df = pd.DataFrame(relative_data)
    return new_df

def process_csv_relative(filepath, output_filepath):
    """
    Đọc file CSV chứa tọa độ tuyệt đối của 13 landmarks (26 cột),
    chuyển đổi sang tọa độ tương đối dựa trên NOSE, và lưu kết quả.
    Kết quả sẽ chỉ chứa 12 landmark (24 cột).
    """
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return

    # Kiểm tra số cột (phải đúng 26)
    if len(df.columns) != 26:
        print(f"File {filepath} không có đúng 26 cột.")
        return

    new_df = convert_to_relative(df)
    new_df.to_csv(output_filepath, index=False)
    print(f"Saved relative landmarks to: {output_filepath}")

# Đường dẫn gốc chứa file CSV (với cấu trúc thư mục nhiều cấp)
input_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split[02]"
# Thư mục output cho file đã chuyển đổi
output_folder = "/content/drive/MyDrive/Gym/data/relative_landmarks"
os.makedirs(output_folder, exist_ok=True)

def process_file(file_path):
    # Giữ nguyên cấu trúc thư mục ban đầu
    relative_path = os.path.relpath(os.path.dirname(file_path), input_folder)
    output_dir = os.path.join(output_folder, relative_path)
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, os.path.basename(file_path))
    process_csv_relative(file_path, output_path)

# Thu thập tất cả các file CSV trong input_folder (bao gồm các subfolder)
all_csv_files = []
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.csv'):
            all_csv_files.append(os.path.join(root, file))

# Xử lý song song các file CSV sử dụng multithreading
num_workers = 16  # Điều chỉnh số worker tùy theo tài nguyên
with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
    executor.map(process_file, all_csv_files)


Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_landmarks/train/barbell biceps curl/barbell biceps curl_2.csv
Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_landmarks/train/barbell biceps curl/barbell biceps curl_31.csv
Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_landmarks/train/barbell biceps curl/barbell biceps curl_25.csv
Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_landmarks/train/barbell biceps curl/barbell biceps curl_5.csv
Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_landmarks/train/barbell biceps curl/barbell biceps curl_18.csv
Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_landmarks/train/barbell biceps curl/barbell biceps curl_30.csv
Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_landmarks/train/barbell biceps curl/barbell biceps curl_24.csv
Saved relative landmarks to: /content/drive/MyDrive/Gym/data/relative_l

In [ ]:
import os
import pandas as pd
import concurrent.futures

def convert_to_relative_33(df):
    """
    Chuyển tọa độ tuyệt đối của 33 landmarks (với cột "Frame" ở đầu)
    thành tọa độ tương đối dựa trên điểm NOSE.

    Input: DataFrame với 133 cột:
      - Cột 1: Frame
      - Sau đó 33 landmarks, mỗi landmark có 4 cột theo thứ tự:
          {LANDMARK}_x, {LANDMARK}_y, {LANDMARK}_z, {LANDMARK}_visibility
    Output: DataFrame mới với:
      - Cột "Frame"
      - 32 landmarks (ngoại trừ NOSE) với các cột:
          {LANDMARK}_x, {LANDMARK}_y, {LANDMARK}_z, {LANDMARK}_visibility
        trong đó các tọa độ x, y, z được tính theo giá trị tuyệt đối trừ đi tọa độ của NOSE.
    """
    # Giả sử cột đầu tiên là "Frame"
    frame_series = df["Frame"]

    # Danh sách tên landmarks theo thứ tự (33 landmark)
    landmark_names = [
        "NOSE",
        "LEFT_EYE_INNER",
        "LEFT_EYE",
        "LEFT_EYE_OUTER",
        "RIGHT_EYE_INNER",
        "RIGHT_EYE",
        "RIGHT_EYE_OUTER",
        "LEFT_EAR",
        "RIGHT_EAR",
        "MOUTH_LEFT",
        "MOUTH_RIGHT",
        "LEFT_SHOULDER",
        "RIGHT_SHOULDER",
        "LEFT_ELBOW",
        "RIGHT_ELBOW",
        "LEFT_WRIST",
        "RIGHT_WRIST",
        "LEFT_PINKY",
        "RIGHT_PINKY",
        "LEFT_INDEX",
        "RIGHT_INDEX",
        "LEFT_THUMB",
        "RIGHT_THUMB",
        "LEFT_HIP",
        "RIGHT_HIP",
        "LEFT_KNEE",
        "RIGHT_KNEE",
        "LEFT_ANKLE",
        "RIGHT_ANKLE",
        "LEFT_HEEL",
        "RIGHT_HEEL",
        "LEFT_FOOT_INDEX",
        "RIGHT_FOOT_INDEX"
    ]

    # Số cột trong file đầu vào: 1 (Frame) + 33*4 = 133
    # Xác định vị trí các cột landmark trong DataFrame (bỏ qua cột "Frame")
    # Ta giả định thứ tự các cột như sau:
    # index 1-4: NOSE, 5-8: LEFT_EYE_INNER, 9-12: LEFT_EYE, ...
    # Chúng ta có thể tính chỉ số bắt đầu của mỗi landmark: landmark i có cột bắt đầu tại: 1 + i*4
    # Lấy tọa độ NOSE:
    nose_idx = 1  # Vì cột "Frame" ở vị trí 0
    nose_x = df.iloc[:, nose_idx]
    nose_y = df.iloc[:, nose_idx + 1]
    nose_z = df.iloc[:, nose_idx + 2]

    relative_dict = {"Frame": frame_series}
    # Duyệt qua các landmark, bỏ qua NOSE
    for i, lm in enumerate(landmark_names):
        if lm == "NOSE":
            continue
        start_col = 1 + i * 4
        # Lấy các cột: x, y, z, visibility
        lm_x = df.iloc[:, start_col]
        lm_y = df.iloc[:, start_col + 1]
        lm_z = df.iloc[:, start_col + 2]
        lm_vis = df.iloc[:, start_col + 3]  # giữ nguyên giá trị visibility

        # Tính tọa độ tương đối dựa trên NOSE
        rel_x = lm_x - nose_x
        rel_y = lm_y - nose_y
        rel_z = lm_z - nose_z

        relative_dict[f"{lm}_x"] = rel_x
        relative_dict[f"{lm}_y"] = rel_y
        relative_dict[f"{lm}_z"] = rel_z
        relative_dict[f"{lm}_visibility"] = lm_vis

    new_df = pd.DataFrame(relative_dict)
    return new_df

def process_csv_relative_33(filepath, output_filepath):
    """
    Đọc file CSV chứa toàn bộ 33 landmarks (với 133 cột: Frame + 33*4),
    chuyển đổi tọa độ tuyệt đối thành tọa độ tương đối dựa trên NOSE,
    và lưu kết quả.

    Kết quả sẽ có 129 cột: "Frame" và 32 landmarks (mỗi landmark có 4 cột).
    """
    try:
        df = pd.read_csv(filepath)
    except Exception as e:
        print(f"Error reading file {filepath}: {e}")
        return

    # Kiểm tra số cột: cần có 133 cột
    if df.shape[1] != 133:
        print(f"File {filepath} không có đúng 133 cột, hiện tại có {df.shape[1]} cột.")
        return

    new_df = convert_to_relative_33(df)
    new_df.to_csv(output_filepath, index=False)
    print(f"Saved relative landmarks to: {output_filepath}")

# Thiết lập đường dẫn thư mục input và output
input_folder = "/content/drive/MyDrive/Gym/data/Merge_data_to_csv_unnormalize_split_full"  # Thư mục chứa file CSV 33 landmarks
output_folder = "/content/drive/MyDrive/Gym/data/relative_landmarks_33"
os.makedirs(output_folder, exist_ok=True)

def process_file(file_path):
    # Giữ nguyên cấu trúc thư mục ban đầu
    relative_path = os.path.relpath(os.path.dirname(file_path), input_folder)
    output_dir = os.path.join(output_folder, relative_path)
    os.makedirs(output_dir, exist_ok=True)
    output_path = os.path.join(output_dir, os.path.basename(file_path))
    process_csv_relative_33(file_path, output_path)

# Thu thập tất cả các file CSV trong input_folder (bao gồm các subfolder)
all_csv_files = []
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.endswith('.csv'):
            all_csv_files.append(os.path.join(root, file))

num_workers = 16  # Có thể điều chỉnh theo tài nguyên hệ thống
with concurrent.futures.ThreadPoolExecutor(max_workers=num_workers) as executor:
    executor.map(process_file, all_csv_files)
